In [ ]:
%pip install -r ../requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import duckdb

import joblib as jlib

from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
con = duckdb.connect("duckdb.ddb")

In [ ]:
data = con.table("").fetchdf()

params = data[["X", "Y", "Z"]]
labels = data["A"]

In [ ]:
def fit(params, labels, order):
    model = SARIMAX(labels, params, order)
    rs = model.fit()

    print(rs.summary())

    return rs

fit(params=params, labels=labels, order=(1,1,1))

In [ ]:
def predict(model, steps: int = 10):
    pass

In [ ]:
def comparePrediction():
    pass

In [ ]:
jlib.dump(model, "")

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def fit_arimax(data, order=(1, 1, 1), exog_cols=['X1', 'X2'], forecast_steps=10):
    """
    Fit an ARIMAX model to the data and make forecasts.
    
    Parameters:
    data (pd.DataFrame): DataFrame containing the time series and exogenous variables.
    order (tuple): The (p,d,q) order of the ARIMA model.
    exog_cols (list): List of column names for the exogenous variables.
    forecast_steps (int): Number of steps to forecast.
    
    Returns:
    pd.DataFrame: DataFrame containing the observed, forecasted values, and confidence intervals.
    """
    exog = data[exog_cols]
    model = SARIMAX(data['Y'], exog=exog, order=order)
    results = model.fit()
    print(results.summary())

    forecast = results.get_forecast(steps=forecast_steps, exog=np.random.randn(forecast_steps, len(exog_cols)))
    forecast_index = pd.date_range(start=data.index[-1] + pd.Timedelta(days=1), periods=forecast_steps)
    forecast_mean = forecast.predicted_mean
    forecast_ci = forecast.conf_int()

    forecast_df = pd.DataFrame({'Forecast': forecast_mean}, index=forecast_index)
    forecast_df['Lower CI'] = forecast_ci.iloc[:, 0]
    forecast_df['Upper CI'] = forecast_ci.iloc[:, 1]
    
    return forecast_df